# 🧠 Teman Bisnis AI — Knowledge Base Builder
Notebook ini membangun & mengelola database vektor (ChromaDB) yang menjadi 'otak' dari RAG chatbot.

**Urutan eksekusi:** Cell 1 → 2 → 3 → 4 → 5 → 6 (Cell 7 opsional untuk reset)

| Cell | Fungsi |
|------|--------|
| 1 | Install & Import |
| 2 | Konfigurasi |
| 3 | Load Dokumen |
| 4 | Chunking |
| 5 | Build ChromaDB |
| 6 | Validasi & Test Query |
| 7 | (Opsional) Reset DB |

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Install dependencies (jalankan sekali)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Uncomment baris di bawah kalau belum install
# !pip install langchain langchain-community langchain-huggingface
# !pip install chromadb sentence-transformers
# !pip install unstructured python-docx openpyxl  # untuk support .docx & .xlsx

import os
import warnings
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import (
    DirectoryLoader, TextLoader, CSVLoader,
    UnstructuredWordDocumentLoader, PyMuPDFLoader
)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document

print('✅ Import selesai.')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Konfigurasi (ubah sesuai kebutuhan)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CONFIG = {
    # Path
    'docs_dir'       : './docs',
    'chroma_dir'     : './chroma_db',

    # Embedding model
    # Opsi lain yang lebih bagus (tapi lebih berat):
    # 'intfloat/multilingual-e5-small'  ← lebih paham Bahasa Indonesia
    # 'paraphrase-multilingual-MiniLM-L12-v2'
    'embed_model'    : 'all-MiniLM-L6-v2',

    # Chunking — naikkan chunk_size kalau dokumen lu naratif panjang
    'chunk_size'     : 1000,
    'chunk_overlap'  : 150,   # overlap lebih besar = konteks lebih nyambung

    # Koleksi ChromaDB (bisa beda koleksi per topik bisnis)
    'collection_name': 'temanbisnis',
}

print('⚙️  Konfigurasi aktif:')
for k, v in CONFIG.items():
    print(f'   {k:20s}: {v}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Load Dokumen (TXT, CSV, DOCX, PDF)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
docs_dir = CONFIG['docs_dir']
all_docs = []
load_summary = []

# Helper
def load_type(glob, loader_cls, label, **kwargs):
    try:
        loader = DirectoryLoader(docs_dir, glob=glob, loader_cls=loader_cls,
                                  loader_kwargs=kwargs, silent_errors=True)
        docs = loader.load()
        load_summary.append(f'  {label:8s}: {len(docs):3d} dokumen')
        return docs
    except Exception as e:
        load_summary.append(f'  {label:8s}: ⚠️  skip ({e})')
        return []

all_docs += load_type('./*.txt',  TextLoader,  'TXT', encoding='utf-8')
all_docs += load_type('./*.csv',  CSVLoader,   'CSV')
all_docs += load_type('./*.docx', UnstructuredWordDocumentLoader, 'DOCX')
all_docs += load_type('./*.pdf',  PyMuPDFLoader, 'PDF')

print(f'📂 Folder: {docs_dir}')
print('\n'.join(load_summary))
print(f'\n📄 Total dokumen dimuat: {len(all_docs)}')

# Preview isi dokumen pertama
if all_docs:
    print(f'\n🔍 Preview dokumen pertama ({all_docs[0].metadata.get("source", "?")})')
    print('   ' + all_docs[0].page_content[:300].replace('\n', '\n   ') + '...')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Chunking + Inspeksi
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG['chunk_size'],
    chunk_overlap=CONFIG['chunk_overlap'],
    separators=['\n\n', '\n', '. ', ' ', ''],  # urutan prioritas pemisah
)

chunks = text_splitter.split_documents(all_docs)

# Statistik
chunk_lens = [len(c.page_content) for c in chunks]
print(f'✂️  Hasil chunking:')
print(f'   Total chunks : {len(chunks)}')
print(f'   Rata-rata len: {sum(chunk_lens)//len(chunk_lens)} karakter')
print(f'   Min / Max    : {min(chunk_lens)} / {max(chunk_lens)} karakter')

print(f'\n🔍 Contoh chunk #1:')
print('   ' + chunks[0].page_content[:400].replace('\n', '\n   '))
print(f'   Metadata: {chunks[0].metadata}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Embedding & Simpan ke ChromaDB
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print(f'🔄 Loading embedding model: {CONFIG["embed_model"]}')
embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG['embed_model'],
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},  # cosine similarity lebih akurat
)

print(f'💾 Menyimpan {len(chunks)} chunks ke ChromaDB...')
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CONFIG['chroma_dir'],
    collection_name=CONFIG['collection_name'],
)

total_in_db = vector_db._collection.count()
print(f'\n✅ Selesai! Database sekarang punya {total_in_db} vectors.')
print(f'   Lokasi: {os.path.abspath(CONFIG["chroma_dir"])}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — Validasi: Test Query ke ChromaDB
# Jalankan ini untuk memastikan RAG bekerja dengan benar
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TEST_QUERIES = [
    'berapa modal yang dibutuhkan untuk rental mobil?',
    'kode KBLI untuk usaha rental kendaraan',
    'perjanjian sewa dan kontrak',
]

# Load ulang DB (simulasi seperti yang dilakukan app.py)
db_check = Chroma(
    persist_directory=CONFIG['chroma_dir'],
    embedding_function=embeddings,
    collection_name=CONFIG['collection_name'],
)

for query in TEST_QUERIES:
    print(f'\n🔎 Query: "{query}"')
    results = db_check.similarity_search_with_score(query, k=2)
    for i, (doc, score) in enumerate(results):
        print(f'   [{i+1}] Score: {score:.4f} | Sumber: {doc.metadata.get("source", "?")}')
        print(f'       {doc.page_content[:200].strip()}...')

print('\n✅ Validasi selesai. Kalau hasilnya relevan, RAG siap dipakai!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 (OPSIONAL) — Reset Database
# ⚠️  HATI-HATI: ini hapus semua data di ChromaDB!
# Jalankan hanya kalau mau rebuild dari awal.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import shutil

CONFIRM_RESET = False  # ← ganti ke True kalau mau reset

if CONFIRM_RESET:
    if os.path.exists(CONFIG['chroma_dir']):
        shutil.rmtree(CONFIG['chroma_dir'])
        print(f'🗑️  ChromaDB dihapus: {CONFIG["chroma_dir"]}')
        print('   Sekarang jalankan Cell 3 → 4 → 5 lagi untuk rebuild.')
    else:
        print('⚠️  Folder tidak ditemukan, tidak ada yang dihapus.')
else:
    print('ℹ️  Reset dibatalkan. Set CONFIRM_RESET = True untuk melanjutkan.')